# Phase 2: Feature Engineering & Model Training 🤖📈

Welcome to the **Model Training & Evaluation** stage. In this phase, we will:
1. Load the cleaned dataset from Phase 1 (`cleaned_dataset.csv`).
2. Select features carefully to prevent data leakage (using lagged neighbor features instead of current ones).
3. Set up a chronological validation split to evaluate time-series forecasting accuracy.
4. Train a Machine Learning model (`RandomForestRegressor`) and log training/testing performance.
5. Plot actual vs. predicted price curves for visual validation.
6. Save the final model for dashboard integration.

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error
import joblib
import os

print("Libraries imported successfully!")

In [ ]:
# Cell 2: Load Cleaned Dataset
csv_path = "cleaned_dataset.csv"
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Please run Phase 1 first to generate '{csv_path}'.")

df = pd.read_csv(csv_path)
print(f"Cleaned dataset loaded. Shape: {df.shape}")

In [ ]:
# Cell 3: Feature Engineering & Leakage Prevention
# EXCLUDE contemporaneous variables: 'neigh_price', 'neigh_price_diff', 'neigh_arrivals', 'neigh_arrival_ratio'
# INCLUDE only lagged neighbor features and weather/time features
features = [
    'Market_id', 'Month_Num', 'Week_Of_Month', 'time_index', 'month_sin', 'month_cos',
    'Rainfall', 'rain_lag_1', 'rain_lag_2', 'rain_lag_4', 'rain_lag_8', 
    'rain_cum_4w', 'rain_cum_8w', 'rain_cum_12w', 'is_monsoon', 'is_dry', 
    'monsoon_total', 'monsoon_prev_year', 'temp_mean', 'temp_min', 'temp_max', 'temp_range', 
    'temp_mean_lag_1', 'temp_mean_lag_2', 'temp_mean_lag_3', 'temp_mean_lag_4', 
    'temp_range_lag_1', 'temp_range_lag_2', 'temp_mean_roll_2', 'temp_mean_roll_4', 
    'has_neighbors', 
    'neigh_price_lag_1', 'neigh_price_lag_2', 'neigh_price_lag_4', 
    'neigh_price_diff_lag_1', 'neigh_price_diff_lag_2', 'neigh_price_diff_lag_4'
]

print(f"Selected {len(features)} features for training.")
print("Feature columns:")
for i, col in enumerate(features):
    print(f"{i+1}. {col}")

In [ ]:
# Cell 4: Chronological Validation Split (Avoiding Data Leakage)
# Since we have 288 weeks, we train on weeks 1 to 240, and test on weeks 241 to 288.
split_week = 240

train_df = df[df['time_index'] <= split_week]
test_df = df[df['time_index'] > split_week]

X_train = train_df[features]
y_train = train_df['Price_Rs_Per_Quintal']
X_test = test_df[features]
y_test = test_df['Price_Rs_Per_Quintal']

print(f"Training set shape: {X_train.shape} (weeks 1 to {split_week})")
print(f"Testing set shape: {X_test.shape} (weeks {split_week+1} to 288)")

In [ ]:
# Cell 5: Model Training
print("Training RandomForestRegressor...")
rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_train = rf.predict(X_train)
y_pred_test = rf.predict(X_test)

train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)
test_mape = mean_absolute_percentage_error(y_test, y_pred_test)

print("\n=== Model Evaluation (Chronological Split) ===")
print(f"Train R2 Score: {train_r2:.4f}")
print(f"Test R2 Score:  {test_r2:.4f}")
print(f"Test MAPE:      {test_mape:.4%}")

In [ ]:
# Cell 6: Feature Importances Analysis
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 6))
sns.barplot(x=importances[indices[:15]], y=np.array(features)[indices[:15]], palette='viridis')
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance Value')
plt.ylabel('Feature Name')
plt.show()

In [ ]:
# Cell 7: Plot Actual vs. Predicted Prices (Sample Market)
sample_market = "Ahmednagar APMC"
market_test_df = test_df[test_df['Market'] == sample_market]

X_sample = market_test_df[features]
y_sample = market_test_df['Price_Rs_Per_Quintal']
y_sample_pred = rf.predict(X_sample)

plt.figure(figsize=(12, 6))
plt.plot(market_test_df['time_index'], y_sample.values, label='Actual Price', color='teal', marker='o', linewidth=2)
plt.plot(market_test_df['time_index'], y_sample_pred, label='Predicted Price (AI)', color='orange', linestyle='--', marker='x', linewidth=2)
plt.title(f'Actual vs. Predicted Price Curve (Test Window) - {sample_market}')
plt.xlabel('Time Index (Weeks)')
plt.ylabel('Price (Rs per Quintal)')
plt.legend()
plt.show()

In [ ]:
# Cell 8: Train Final Model on All Data & Save
print("Training the final production model on all 288 weeks...")
rf_final = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf_final.fit(df[features], df['Price_Rs_Per_Quintal'])

model_filename = "price_forecast_rf.joblib"
joblib.dump(rf_final, model_filename)
print(f"Final model successfully trained and serialized to '{model_filename}'!")